# Healthcare KG — Population Intelligence V2
**Isolated Colab research notebook · NHANES 2017–2018 · no GitHub writes**

**Research objective:** Reassess V1 CKD + probable T2D screening population using complex survey design; segment by kidney function, albuminuria, age, sex and race/ethnicity; describe **observable Kerendia-related measurements**, not eligibility.

**Scientific guardrails:** A single eGFR/UACR measurement does **not** confirm CKD chronicity. Age at diabetes diagnosis is not a definitive T2D diagnosis. NHANES is cross-sectional and excludes institutionalized persons. No result here is an FDA-label-based eligibility count or a market forecast. Do not add results to the production graph.

In [ ]:
%pip -q install pandas numpy scipy requests networkx pyreadstat

## 1 · Isolated workspace and baseline archive
Upload the original `24OCT26_Healthcare_KG_Archive.zip`. This notebook never connects to GitHub.

In [ ]:
from pathlib import Path
import zipfile, hashlib, json, datetime, shutil
from google.colab import files
ROOT=Path('/content/healthcare_kg_population_v2'); ROOT.mkdir(parents=True,exist_ok=True)
uploaded=files.upload()
archive=next((Path('/content')/k for k in uploaded if k.lower().endswith('.zip')),None)
assert archive is not None, 'Upload the October 24 KG ZIP.'
with zipfile.ZipFile(archive) as z:
    names=[x for x in z.namelist() if x.endswith('.json') and 'medical_kg' in x and not x.startswith('__MACOSX')]
    assert len(names)==1, f'Expected one KG JSON, got {names}'
    original=z.read(names[0])
kg_path=ROOT/'baseline_kg_COPY.json'; kg_path.write_bytes(original)
base_hash=hashlib.sha256(original).hexdigest()
kg=json.loads(original)
print('Baseline SHA256:',base_hash)
print('Top-level KG keys:',list(kg)[:12])

## 2 · Retrieve and cache CDC NHANES files
2017–2018 public-use XPT: DEMO_J, ALB_CR_J, BIOPRO_J, DIQ_J. CDC source URLs and SHA256 recorded. No current-cycle data are mixed into this baseline.

In [ ]:
import pandas as pd, numpy as np, requests
BASE='https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/'
SOURCES={name:BASE+name+'.xpt' for name in ['DEMO_J','ALB_CR_J','BIOPRO_J','DIQ_J']}
def load_xpt(name,url):
    path=ROOT/(name+'.xpt')
    if not path.exists():
        response=requests.get(url,timeout=120,headers={'User-Agent':'Healthcare-KG-Research/2.0'})
        response.raise_for_status()
        assert len(response.content)>1000, f'Unexpected download: {name}'
        path.write_bytes(response.content)
    return pd.read_sas(path,format='xport')
tables={k:load_xpt(k,v) for k,v in SOURCES.items()}
for k,v in tables.items(): print(k,len(v),'records')

## 3 · Construct analytical cohort and clinical screening indicators

- Adult: ≥20 years.
- eGFR: 2021 CKD-EPI creatinine equation (race-free).
- UACR (mg/g): urine albumin (µg/mL) × 100 ÷ urine creatinine (mg/dL).
- CKD screen: eGFR <60 or UACR ≥30, **only when both measurements are available**.
- V1-comparable T2D proxy: reported diabetes and diagnosis age ≥30. A **sensitivity analysis** also uses age at diagnosis ≥20, which is less specific.
- Serum potassium is reported as a descriptive measurement, **not** a prescribing decision.
- eGFR groups: G1 ≥90; G2 60–89; G3a 45–59; G3b 30–44; G4 15–29; G5 <15.
- UACR groups: A1 <30; A2 30–299; A3 ≥300.


In [ ]:
d=tables['DEMO_J'][['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU']].copy()
for name in ['ALB_CR_J','BIOPRO_J','DIQ_J']:
    d=d.merge(tables[name],on='SEQN',how='left',validate='one_to_one')
required=['RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU','URXUMA','URXUCR','LBXSCR','DIQ010','DIQ040']
assert not [c for c in required if c not in d], 'Missing required NHANES variables'
d=d.loc[d.RIDAGEYR>=20].copy()
for col in required: d[col]=pd.to_numeric(d[col],errors='coerce')
female=d.RIAGENDR.eq(2); kappa=np.where(female,.7,.9); alpha=np.where(female,-.241,-.302)
r=d.LBXSCR/kappa
d['egfr']=142*np.minimum(r,1)**alpha*np.maximum(r,1)**(-1.2)*.9938**d.RIDAGEYR*np.where(female,1.012,1)
d.loc[d.LBXSCR.isna()|d.LBXSCR.le(0),'egfr']=np.nan
d['uacr']=100*d.URXUMA/d.URXUCR
d.loc[d.URXUCR.isna()|d.URXUCR.le(0),'uacr']=np.nan
d['ckd_screen']=np.where(d.egfr.notna() & d.uacr.notna(),(d.egfr<60)|(d.uacr>=30),np.nan)
# DIQ010: 1 yes, 2 no, 3 borderline; DIQ040: age first told diabetes.
d['diabetes_reported']=np.where(d.DIQ010.isin([1,2]),d.DIQ010.eq(1),np.nan)
def diabetes_proxy(min_age):
    return np.where(d.diabetes_reported.eq(0),0,
        np.where(d.diabetes_reported.eq(1)&d.DIQ040.between(1,100),(d.DIQ040>=min_age).astype(float),np.nan))
for threshold,label in [(30,'t2d_proxy_v1'),(20,'t2d_proxy_sensitivity')]:
    d[label]=diabetes_proxy(threshold)
    d['joint_'+label]=np.where(d.ckd_screen.notna() & pd.notna(d[label]),
                                    (d.ckd_screen==1)&(d[label]==1),np.nan)
d['egfr_category']=pd.cut(d.egfr,bins=[-np.inf,15,30,45,60,90,np.inf],right=False,
    labels=['G5 <15','G4 15-29','G3b 30-44','G3a 45-59','G2 60-89','G1 >=90'])
d['uacr_category']=pd.cut(d.uacr,bins=[0,30,300,np.inf],right=False,labels=['A1 <30','A2 30-299','A3 >=300'])
d['age_group']=pd.cut(d.RIDAGEYR,[20,45,65,75,np.inf],right=False,labels=['20-44','45-64','65-74','75+'])
d['sex']=d.RIAGENDR.map({1:'Male',2:'Female'})
d['race_ethnicity']=d.RIDRETH3.map({1:'Mexican American',2:'Other Hispanic',3:'Non-Hispanic White',4:'Non-Hispanic Black',6:'Non-Hispanic Asian',7:'Other/multiracial'})
# BIOPRO_J typically includes LBXSKSI (serum potassium mmol/L); fail gracefully if absent.
d['potassium_mmol_l']=pd.to_numeric(d['LBXSKSI'],errors='coerce') if 'LBXSKSI' in d else np.nan
print('Adults:',len(d),'V1 joint evaluable:',d.joint_t2d_proxy_v1.notna().sum(),
      'V1 joint positive:',int((d.joint_t2d_proxy_v1==1).sum()))

## 4 · Design-based estimates and confidence intervals

Taylor linearization with NHANES masked strata and PSUs, including out-of-domain observations as zero contributions. Ratio estimates use a linearized residual. The **complete-case population denominator is not automatically the entire U.S. adult population**; both it and the missingness fraction are reported. Suppress interpretive use of small/unreliable domains (unweighted positive count <30). Confidence intervals are approximate and should be externally benchmarked before publication.

In [ ]:
from scipy.stats import t as student_t
base_valid=d.WTMEC2YR.gt(0)&d.SDMVSTRA.notna()&d.SDMVPSU.notna()
def variance_total(z,contrib):
    t=z[['SDMVSTRA','SDMVPSU']].copy(); t['v']=z.WTMEC2YR*contrib.astype(float)
    g=t.groupby(['SDMVSTRA','SDMVPSU'],observed=True).v.sum()
    variance=0.; nstrata=0; npsu=0
    for _,block in g.groupby(level=0):
        vals=block.to_numpy(); m=len(vals)
        if m>=2:
            variance+=m/(m-1)*np.square(vals-vals.mean()).sum()
            nstrata+=1; npsu+=m
    return variance,max(1,npsu-nstrata)
def estimate(domain,case,definition,segment):
    # All participants with valid design variables remain in the variance calculation.
    z=d.loc[base_valid].copy()
    dom=domain.reindex(z.index).fillna(False).astype(bool)
    valid=case.reindex(z.index).notna()
    y=(dom&valid&case.reindex(z.index).eq(1)).astype(float)
    x=(dom&valid).astype(float)
    count=float((z.WTMEC2YR*y).sum()); denom=float((z.WTMEC2YR*x).sum())
    v_count,dfree=variance_total(z,y)
    p=count/denom if denom>0 else np.nan
    resid=y-p*x if denom>0 else y*0
    v_resid,_=variance_total(z,resid)
    se_p=np.sqrt(v_resid)/denom if denom>0 else np.nan
    crit=student_t.ppf(.975,dfree)
    n=int((dom&valid).sum()); cases=int(y.sum())
    return dict(definition=definition,segment=segment,unweighted_n=n,unweighted_cases=cases,
        weighted_complete_case_denominator=round(denom),weighted_cases=round(count),
        count_ci_low=max(0,round(count-crit*np.sqrt(v_count))),count_ci_high=round(count+crit*np.sqrt(v_count)),
        prevalence=p,prevalence_ci_low=max(0,p-crit*se_p) if denom else np.nan,
        prevalence_ci_high=min(1,p+crit*se_p) if denom else np.nan,
        degrees_freedom=dfree,unstable_flag=(cases<30 or n<100),
        excluded_missing_n=int((dom&~valid).sum()))
all_adults=pd.Series(True,index=d.index)
rows=[]
for key in ['joint_t2d_proxy_v1','joint_t2d_proxy_sensitivity']:
    case=d[key]
    rows.append(estimate(all_adults,case,key,'All adults'))
    for col in ['age_group','sex','race_ethnicity']:
        for val in d[col].dropna().unique():
            rows.append(estimate(d[col].eq(val),case,key,f'{col}: {val}'))
summary=pd.DataFrame(rows)
summary.to_csv(ROOT/'v2_population_estimates.csv',index=False)
display(summary[['definition','segment','unweighted_cases','weighted_cases','prevalence','unstable_flag']])

## 5 · Mutually exclusive kidney categories and observable measurements

Among persons meeting the V1 joint proxy, generate G × A cells and potassium bands. These are **descriptive** and **not** a Kerendia-eligible population. Low cell counts must not be interpreted as reliable estimates. A potassium value alone cannot establish treatment suitability; additional clinical history, repeated testing, indication, contraindications, and prescriber judgment are required.

In [ ]:
joint=d.joint_t2d_proxy_v1.eq(1)
segments=[]
for g in d.egfr_category.cat.categories:
    for a in d.uacr_category.cat.categories:
        flag=(joint & d.egfr_category.eq(g) & d.uacr_category.eq(a)).astype(float)
        segments.append(estimate(all_adults,flag,'GxA joint proxy',f'{g} | {a}'))
for label,mask in [('Potassium <5.0',d.potassium_mmol_l.lt(5.0)),
                   ('Potassium >=5.0',d.potassium_mmol_l.ge(5.0)),
                   ('Potassium missing',d.potassium_mmol_l.isna())]:
    segments.append(estimate(all_adults,(joint&mask).astype(float),'Potassium descriptive',label))
seg=pd.DataFrame(segments)
seg.to_csv(ROOT/'v2_kidney_segmentation.csv',index=False)
display(seg[['definition','segment','unweighted_cases','weighted_cases','unstable_flag']])
print('GxA observed joint-proxy cases:',seg.loc[seg.definition=='GxA joint proxy','unweighted_cases'].sum())
print('Joint-proxy unweighted cases:',int(joint.sum()))
assert seg.loc[seg.definition=='GxA joint proxy','unweighted_cases'].sum()==int(joint.sum()), 'GxA cells do not partition joint cases; investigate missingness' 

## 6 · Quality checks, source manifest and provenance

The validation report explicitly checks V1 comparability (where available), partition sums, missingness, and baseline JSON integrity. **No statistical estimate is written into the baseline KG.**

In [ ]:
import hashlib
manifest=pd.DataFrame([dict(dataset=k,source_url=url,sha256=hashlib.sha256((ROOT/(k+'.xpt')).read_bytes()).hexdigest(),
    bytes=(ROOT/(k+'.xpt')).stat().st_size,cycle='2017-2018') for k,url in SOURCES.items()])
manifest.to_csv(ROOT/'v2_source_manifest.csv',index=False)
validation=dict(baseline_sha256=base_hash,baseline_unchanged=hashlib.sha256(kg_path.read_bytes()).hexdigest()==base_hash,
    adults=len(d),joint_proxy_evaluable=int(d.joint_t2d_proxy_v1.notna().sum()),
    joint_proxy_cases=int(joint.sum()),
    missing_joint_proxy=int(d.joint_t2d_proxy_v1.isna().sum()),
    gxa_partition_cases=int(seg.loc[seg.definition=='GxA joint proxy','unweighted_cases'].sum()),
    population_estimate=int(summary.loc[(summary.definition=='joint_t2d_proxy_v1')&(summary.segment=='All adults'),'weighted_cases'].iloc[0]),
    potassium_available=('LBXSKSI' in d.columns),
    timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
    caveats=['one-time CKD screen','T2D proxy, not confirmed type','complete-case estimate','approximate design-based CI','not Kerendia eligibility'],
    github_modified=False)
assert validation['baseline_unchanged']
assert validation['gxa_partition_cases']==validation['joint_proxy_cases']
(ROOT/'v2_validation_report.json').write_text(json.dumps(validation,indent=2))
print(json.dumps(validation,indent=2))

## 7 · Optional durable Google Drive export
Only copies derived outputs and baseline JSON copy to a **new V2 folder**. No source repository access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
target=Path('/content/drive/MyDrive/Healthcare_KG_Population_Intelligence_V2'); target.mkdir(parents=True,exist_ok=True)
for name in ['v2_population_estimates.csv','v2_kidney_segmentation.csv','v2_source_manifest.csv','v2_validation_report.json','baseline_kg_COPY.json']:
    shutil.copy2(ROOT/name,target/name)
print('Saved:',target)

## Acceptance and V3 gate

**Review before accepting V2:** (1) compare V1 vs V2 national estimate and explain differences; (2) inspect complete-case bias; (3) benchmark NHANES design variance against an independent survey-analysis implementation; (4) inspect small G×A cells; (5) verify clinical definitions against current FDA labeling before making eligibility claims. Only after these checks should we add newer NHANES cycles or drug opportunity modeling.

**Deliverables:** `v2_population_estimates.csv`, `v2_kidney_segmentation.csv`, `v2_source_manifest.csv`, `v2_validation_report.json`.